In [1]:
import pandas as pd
from fyers_apiv3 import fyersModel


CLIENT_ID = 'G80JG7X1EA-200'

with open('access_token.txt') as f:
    access_token = f.read().strip()

fyers = fyersModel.FyersModel(client_id=CLIENT_ID, token=access_token, is_async=False, log_path='')


def get_option_chain(symbol='NSE:NIFTY50-INDEX', strike_count=10, expiry='', save_path='nifty_option_chain.csv'):
    # strike_count = number of strikes above and below ATM
    # expiry = '' for nearest expiry, or an 'expiry' value from expiryData (epoch string)
    response = fyers.optionchain(data={'symbol': symbol, 'strikecount': strike_count, 'timestamp': expiry})
    if response.get('s') != 'ok':
        raise SystemExit(f'Failed to fetch option chain: {response}')

    data = response['data']
    chain = pd.DataFrame(data['optionsChain'])

    # First row is the underlying itself (strike_price = -1)
    spot = chain.loc[chain['strike_price'] == -1, 'ltp'].iloc[0]
    options = chain[chain['strike_price'] > 0]

    # Put CE and PE of each strike side by side
    cols = ['symbol', 'ltp', 'bid', 'ask', 'volume', 'oi', 'oich']
    ce = options[options['option_type'] == 'CE'].set_index('strike_price')[cols].add_prefix('CE_')
    pe = options[options['option_type'] == 'PE'].set_index('strike_price')[cols].add_prefix('PE_')
    df = ce.join(pe, how='outer').sort_index().reset_index()

    print(f"{symbol}  Spot: {spot}   Expiries: {[e['date'] for e in data['expiryData'][:5]]} ...")
    print(f"Total Call OI: {data['callOi']:,}   Total Put OI: {data['putOi']:,}   PCR: {data['putOi'] / data['callOi']:.2f}")

    df.to_csv(save_path, index=False)
    print(f'Saved {len(df)} strikes to {save_path}')
    return df, data['expiryData']


chain_df, expiries = get_option_chain('NSE:NIFTY50-INDEX', strike_count=10)
chain_df


NSE:NIFTY50-INDEX  Spot: 22581.05   Expiries: ['13-10-2026', '19-10-2026', '27-10-2026', '03-11-2026', '09-11-2026'] ...
Total Call OI: 152,533,615   Total Put OI: 128,419,005   PCR: 0.84
Saved 21 strikes to nifty_option_chain.csv


,strike_price,CE_symbol,CE_ltp,CE_bid,CE_ask,CE_volume,CE_oi,CE_oich,PE_symbol,PE_ltp,PE_bid,PE_ask,PE_volume,PE_oi,PE_oich
0,22100,NSE:NIFTY26O1322100CE,516.60,516.65,517.95,184145.0,112840.0,34775.0,NSE:NIFTY26O1322100PE,22.20,22.10,22.15,38701325.0,4176900.0,1759550.0
1,22150,NSE:NIFTY26O1322150CE,470.00,471.55,473.25,107315.0,42900.0,29510.0,NSE:NIFTY26O1322150PE,27.10,27.00,27.10,25046190.0,1460485.0,816270.0
2,22200,NSE:NIFTY26O1322200CE,428.60,427.95,429.15,975260.0,372775.0,97240.0,NSE:NIFTY26O1322200PE,32.90,32.80,32.95,64793820.0,6966765.0,4053405.0
3,22250,NSE:NIFTY26O1322250CE,384.00,385.20,386.40,229710.0,67405.0,28275.0,NSE:NIFTY26O1322250PE,40.10,39.95,40.10,32557395.0,1703065.0,981175.0
4,22300,NSE:NIFTY26O1322300CE,343.40,344.00,344.85,2529540.0,341510.0,115115.0,NSE:NIFTY26O1322300PE,48.80,48.60,48.75,65848575.0,4483505.0,1347185.0
5,22350,NSE:NIFTY26O1322350CE,304.05,304.60,305.45,976625.0,151840.0,90025.0,NSE:NIFTY26O1322350PE,59.15,59.15,59.30,39077480.0,1561300.0,877305.0
6,22400,NSE:NIFTY26O1322400CE,266.85,267.30,268.05,7436195.0,958555.0,391300.0,NSE:NIFTY26O1322400PE,71.75,71.70,71.75,87390680.0,4978740.0,2387060.0
7,22450,NSE:NIFTY26O1322450CE,232.10,232.00,232.60,4508270.0,383565.0,299260.0,NSE:NIFTY26O1322450PE,86.45,86.20,86.40,66677000.0,2166190.0,370760.0
8,22500,NSE:NIFTY26O1322500CE,198.85,198.90,199.40,34978450.0,3269760.0,1934080.0,NSE:NIFTY26O1322500PE,103.40,103.20,103.40,138955830.0,7130240.0,2655250.0
9,22550,NSE:NIFTY26O1322550CE,168.80,168.80,169.20,35371960.0,2495025.0,2179840.0,NSE:NIFTY26O1322550PE,123.40,123.10,123.40,95256200.0,3573700.0,2638610.0
